In [ ]:
!pip -q install torchio monai
import os, sys
print('Done')

In [ ]:
!git clone https://github.com/MIC-DKFZ/MedNeXt.git mednext
%cd mednext
!pip -q install -e .
%cd ..

sys.path.append('mednext')

In [ ]:
import torch
import torchio as tio
from torch.utils.data import DataLoader, Subset
from torchio.data import SubjectsLoader, SubjectsDataset
import torch.nn as nn
import matplotlib.pyplot as plt
from nnunet_mednext import create_mednext_v1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)
from glob import glob
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [ ]:
!tar -xf /kaggle/input/brats-2021-task1/BraTS2021_Training_Data.tar

In [ ]:
# from pathlib import Path
# import os, glob
# root_path = "/kaggle/working/"

# # print(os.path.join(patient_dir))

# def get_modality_path(patient_dir, modality):
#     patient_dir = os.path.join(root_path, patient_dir)
#     try:
#         return glob.glob(patient_dir + f"/*{modality}.nii.gz")[0]
#     except IndexError:
#         print(f'skipping {patient_dir}')
#         return False

# # print(get_modality_path("/kaggle/working/BraTS2021_00000", "t1"))
# subjects = []

# for patient_dir in os.listdir(root_path):
#     for i in ['t1', 't2', 't1ce', 'flair', 'seg']:
#         if not get_modality_path(patient_dir, i):
#             continue
#         else:
#             subject = tio.Subject(
#                 t1=tio.ScalarImage(get_modality_path(patient_dir, "t1")),
#                 t2=tio.ScalarImage(get_modality_path(patient_dir, "t2")),
#                 t1ce=tio.ScalarImage(get_modality_path(patient_dir, "t1ce")),
#                 flair=tio.ScalarImage(get_modality_path(patient_dir, "flair")),
#                 label=tio.LabelMap(get_modality_path(patient_dir, "seg")),
#             )
#             subjects.append(subject)

In [ ]:
class BratsDataset(SubjectsDataset):
    def __init__(self, root_dir: str, transform=None):
        self.patient_dirs = os.listdir(root_dir)
        self.transform = transform
        self.subjects = []

        for patient_dir in self.patient_dirs:
            t1_path = glob(os.path.join(root_dir, patient_dir, "*t1.nii*"), recursive=True)
            t1ce_path = glob(os.path.join(root_dir, patient_dir, "*t1ce.nii*"), recursive=True)
            t2_path = glob(os.path.join(root_dir, patient_dir, "*t2.nii*"), recursive=True)
            flair_path = glob(os.path.join(root_dir, patient_dir, "*flair.nii*"), recursive=True)
            seg_path = glob(os.path.join(root_dir, patient_dir, "*seg.nii*"), recursive=True)

            if not (t1_path and t1ce_path and t2_path and flair_path and seg_path):
                print(f"Skipping {patient_dir}")
                continue
                
            self.subjects.append(
                 tio.Subject(
                    vol=tio.ScalarImage([t1ce_path[0], flair_path[0]]),
                    label=tio.LabelMap(seg_path[0]),
                )
            )
        super().__init__(self.subjects, transform=transform)

In [ ]:
# root_dir = "/kaggle/working"
# for patient_dir in os.listdir(root_dir):
#     t1_path = glob(os.path.join(root_dir, patient_dir, "*t1.nii.gz*"), recursive=True)
#     # print(os.path.join(root_dir, patient_dir))
#     print(t1_path)

In [ ]:
preprocess = tio.Compose([
    tio.RescaleIntensity(out_min_max=(0, 99.5), exclude=["label"]), 
    tio.ZNormalization(exclude=['label']),
    tio.Resample(1.0),  
    tio.CropOrPad((128, 128, 32)), 
    tio.ZNormalization(exclude=['label']),
])

train_transform = preprocess

dataset = BratsDataset("/kaggle/working", train_transform)

subset_size = 100
subset_indices = torch.randperm(len(dataset))[:subset_size]
subset_dataset = Subset(dataset, subset_indices)

train_len = int(0.7 * subset_size)
val_len = int(0.2 * subset_size)
test_len = subset_size - (train_len + val_len)

train_set, val_set, test_set = torch.utils.data.random_split(subset_dataset, [train_len, val_len, test_len])

train_loader = SubjectsLoader(train_set, batch_size=2, shuffle=True, num_workers=2)
val_loader = SubjectsLoader(val_set, batch_size=2, shuffle=False, num_workers=2)
test_loader = SubjectsLoader(test_set, batch_size=2, shuffle=False, num_workers=2)

In [ ]:
# from nnunet_mednext import create_mednext_v1

# class Mednext(nn.Module):
#     def __init__(self, input_channels=1, output_channels=1):
#         super(MedNextGenerator3D, self).__init__()
        
#         self.model = create_mednext_v1(
#             num_input_channels=input_channels,
#             num_classes=output_channels,
#             model_id='s',
#             kernel_size=3,
#             deep_supervision=False
#         )

#         self.final_activation = nn.Sigmoid()

#     def forward(self, x):
#         x = self.model(x)
#         return self.final_activation(x)
 
# model = Mednext(input_channels=4, output_channels=5)
# sample_input = torch.randn(1, 4, 16, 64, 64)
# output = model(sample_input)
# print("Output shape:", output.shape)

In [ ]:
batch = next(iter(train_loader))
vol_data = batch['vol'][tio.DATA]
sample_img = vol_data[0, 0, :, :, 16]
mask_data = batch['label'][tio.DATA]
sample_mask = mask_data[0, 0, :, :, 16]
# mask_data = np.where(mask_data == 4, 3, mask_data)
# np.unique(mask_data)

plt.subplot(1,2,1)
plt.axis('off')
plt.imshow(sample_img, cmap='gray')
plt.subplot(1, 2, 2)
plt.savefig('sample_train_input.png', dpi=300)
plt.axis('off')
plt.imshow(sample_mask, cmap='gray')

In [ ]:
model = create_mednext_v1(
    num_input_channels=2,
    num_classes=4,
    model_id='S',
    kernel_size=3,
    deep_supervision=False
)

In [ ]:
from monai.metrics import HausdorffDistanceMetric
from monai.metrics import DiceMetric

hd95_metric = HousdorffDistanceMetric(
    include_backgroud=False,
    percentile=95,
    reduction="mean_batch",
    get_not_nans=True
)
dice_metric = DiceMetric(include_backdround=False, reduction="mean_batch")

In [ ]:
hd95_metric.reset()

In [ ]:
from monai.losses import DiceLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
model.to(device)

num_epochs = 3

patience = 5
best_val_loss = float("inf")
epochs_no_improve = 0
early_stop = False

train_losses = []
val_losses = []
metrics = {
    "hd95": [],
    "dice_metric": []
}

for epoch in range(num_epochs):
    model.train()
    epoch_train_loss = 0.0

    for batch in train_loader:
        inputs = batch["vol"][tio.DATA].to(device).float()

        labels = batch["label"][tio.DATA].long().to(device)
        labels = torch.where(labels == 4, torch.tensor(3, device=labels.device), labels)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        epoch_train_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    epoch_val_loss = 0.0
    dice_metric.reset()
    hd95_metric.reset()
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch["vol"][tio.DATA].to(device).float()
            labels = batch["label"][tio.DATA].long().to(device)
            labels = torch.where(labels == 4, torch.tensor(3, device=labels.device), labels)

            outputs = model(inputs)
            val_loss = criterion(outputs, labels)
            epoch_val_loss += val_loss.item()
            
            outputs = torch.softmax(outputs, dim=1)
            outputs = torch.argmax(outputs, dim=1)

            outputs = torch.nn.functional.one_hot(outputs, num_classes=4).permute(0, 3, 1, 2).float() 
            
            labels = torch.nn.functional.one_hot(labels.long(), num_classes=4).permute(0, 3, 1, 2)
            
            hd95_metric(outputs, labels)
            dice_metric(outputs, labels)
            
    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    metrics["hd95"].append(hd95_metric.aggregate())
    metrics["dice_metric"].append(dice_metric.aggregate())

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    scheduler.step()

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at {epoch+1}s")
            break

In [ ]:
np.mean(hd95_values)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(train_losses, 'bo-', label='Training Loss')
plt.plot(val_losses, 'ro-', label='Validation Loss')
plt.title('Training and Validation Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.savefig('losses.png', dpi=300)

plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt

sample = test_set[0] 

image = sample['vol'].data  
label = sample['label'].data 

input_tensor = image.unsqueeze(0).to(device).float()

model.eval()
with torch.no_grad():
    output = model(input_tensor)  

    probs = torch.softmax(output, dim=1)   
    pred = torch.argmax(probs, dim=1)
    
    pred = pred.cpu().squeeze(0)  
    
slice_idx = image.shape[-1] // 2 

# output.squeeze(0)[0, :, :, slice_idx].shape
input_slice = image.cpu()[0, :, :, slice_idx]
label_slice = label.cpu()[0, :, :, slice_idx]
pred_slice = pred.cpu()[:, :, slice_idx]

print(input_slice.shape)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(input_slice, cmap='gray')
axes[0].set_title('Input Image')
axes[0].axis('off')

axes[1].imshow(label_slice, cmap='gray')
axes[1].set_title('Ground Truth')
axes[1].axis('off')

axes[2].imshow(pred_slice, cmap='gray')
axes[2].set_title('Prediction')
axes[2].axis('off')

plt.show()


In [ ]:
# trying with best model

model = model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
# model.to(device)
model.eval()
with torch.no_grad():
    output = model(input_tensor)  

    probs = torch.softmax(output, dim=1)   
    pred = torch.argmax(probs, dim=1)
    
    pred = pred.cpu().squeeze(0)  
    
slice_idx = image.shape[-1] // 2 

# output.squeeze(0)[0, :, :, slice_idx].shape

input_slice = image.cpu().squeeze(0)[:, :, slice_idx]
label_slice = label.cpu().squeeze(0)[:, :, slice_idx]
pred_slice = pred.cpu().squeeze(0)[:, :, slice_idx]

print(torch.where(pred_slice==1))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(input_slice, cmap='gray')
axes[0].set_title('Input Image (T1)')
axes[0].axis('off')

axes[1].imshow(label_slice, cmap='gray')
axes[1].set_title('Ground Truth')
axes[1].axis('off')

axes[2].imshow(pred_slice, cmap='gray')
axes[2].set_title('Prediction')
axes[2].axis('off')

plt.show()
